# 🛋️ AI Interior Designer — سرور مدل روی Colab (نسخه v2)

این نسخه معماری جدیدیه: پایه‌اش نوت‌بوک بازنویسی‌شدهٔ هم‌گروهی (`ai-interior-designer-v2-backend-r3.ipynb`) است — انتخاب دقیق ناحیه (region_id/mask/bbox/point)، ذخیرهٔ تصاویر با شناسه (`image_id`)، حذف شیء (`/delete-object`)، و تشخیص ترکیبی YOLO+SAM+SegFormer. روی این پایه، کارهای قبلی خودمون اضافه شده: دو مدل Fast(SD1.5)/Quality(SDXL)، چک‌پوینت‌های فوتورئالیستی (Realistic Vision / Juggernaut XL)، پایپ‌لاین Img2Img (نه فقط لبه)، افزودن شیء دقیق از روی عکس مرجع (IP-Adapter)، و امنیت اتصال با `CONNECTION_KEY`.

قرارداد API نسبت به نسخهٔ قبلی فرق داره (`selection`، `image_id`، `/capabilities`، `/delete-object`) ولی فرانت فعلی بدون تغییر کار می‌کنه — روت‌ها یه حالت سازگاری هم دارن (اگه بجای `selection`، فقط اسم شیء بفرستی، خودش شیء رو پیدا و انتخاب می‌کنه).

## ۱) بررسی GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU first."
print(torch.cuda.get_device_name(0))


## ۲) نصب کتابخانه‌ها

In [ ]:
import subprocess, sys, os
os.environ["USE_TF"] = "0"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.37.0", "transformers==4.57.6", "accelerate==1.12.0",
    "flask", "flask-cors", "pyngrok", "ultralytics", "segment-anything",
    "opencv-python-headless", "safetensors"])


## ۳) آماده‌سازی مسیرهای ذخیره‌سازی

In [ ]:
import os, torch

# نوت‌بوک تیمی برای Kaggle نوشته شده بود (/kaggle/working)؛ چون زیرساخت ما (ngrok،
# CONNECTION_KEY، فیلد colab_url در فرانت) کاملاً روی Colab استانداردسازی شده،
# پیش‌فرض همون مسیرهای Colab است ولی اگه محیط Kaggle باشه هم خودش تشخیص می‌ده.
CACHE_DIR = "/content/interior_v2/models" if os.path.isdir("/content") else "/kaggle/working/models"
OUTPUT_DIR = "/content/interior_v2/outputs" if os.path.isdir("/content") else "/kaggle/working/outputs"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("CACHE_DIR:", CACHE_DIR)


## ۴) بارگذاری مدل‌ها — دو مدل، لود تنبل

هم SD1.5 (سریع) هم SDXL (باکیفیت‌تر) در دسترسن ولی فقط یکی در هر لحظه واقعاً روی حافظه می‌مونه — `ensure_model()` موقع سوییچ، مدل قبلی رو آزاد می‌کنه.

In [ ]:
import gc
from diffusers import (
    ControlNetModel, AutoencoderKL, UniPCMultistepScheduler,
    StableDiffusionControlNetImg2ImgPipeline, StableDiffusionInpaintPipeline,
    StableDiffusionXLControlNetImg2ImgPipeline, AutoPipelineForInpainting,
)

style_pipe = None
inpaint_pipe = None
_current_model = {"name": None}

# SDXL's ControlNet is much more sensitive to conditioning_scale (0.5 is its own
# model card's recommendation) than SD1.5's (tuned at 1.05 for this project).
GEN_PARAMS = {
    "fast":    {"guidance_scale": 9.0, "controlnet_conditioning_scale": 1.05},
    "quality": {"guidance_scale": 7.0, "controlnet_conditioning_scale": 0.5},
}

# Community fine-tunes instead of vanilla base checkpoints — per
# https://stable-diffusion-art.com/models/ (Realistic Vision for SD1.5,
# Juggernaut XL for SDXL are that guide's own top photorealism picks). Both are
# drop-in replacements: same UNet architecture as their base, so ControlNet
# weights trained on the vanilla base stay fully compatible.
SD15_CHECKPOINT = "SG161222/Realistic_Vision_V6.0_B1_noVAE"
SD15_VAE = "stabilityai/sd-vae-ft-mse"
SDXL_CHECKPOINT = "RunDiffusion/Juggernaut-XL-v9"


def _load_sd15():
    global style_pipe, inpaint_pipe
    print(f"Loading {SD15_CHECKPOINT} (Fast mode)...")
    cn = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_canny", torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    vae15 = AutoencoderKL.from_pretrained(SD15_VAE, torch_dtype=torch.float16, cache_dir=CACHE_DIR)
    # Img2Img: می‌گیره هم عکس واقعی اتاق (image) هم نقشهٔ لبه (control_image) —
    # نسخهٔ قدیمی فقط لبه می‌دید، هیچ‌وقت رنگ/نور/بافت واقعی عکس رو نمی‌دید.
    style_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        SD15_CHECKPOINT, controlnet=cn, vae=vae15, torch_dtype=torch.float16,
        safety_checker=None, cache_dir=CACHE_DIR
    )
    style_pipe.scheduler = UniPCMultistepScheduler.from_config(style_pipe.scheduler.config)
    style_pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
    style_pipe.set_ip_adapter_scale(0.0)
    style_pipe.enable_model_cpu_offload()
    style_pipe.enable_vae_tiling()

    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-inpainting",
        torch_dtype=torch.float16, safety_checker=None, cache_dir=CACHE_DIR
    )
    inpaint_pipe.enable_model_cpu_offload()
    print(f"✅ {SD15_CHECKPOINT} ready (fast, photorealism-tuned, Img2Img)")


def _load_sdxl():
    global style_pipe, inpaint_pipe
    print(f"Loading {SDXL_CHECKPOINT} (Quality mode)... (bigger — first time downloads more)")
    cn = ControlNetModel.from_pretrained(
        "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    vae = AutoencoderKL.from_pretrained(
        "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    style_pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
        SDXL_CHECKPOINT, controlnet=cn, vae=vae, torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    style_pipe.scheduler = UniPCMultistepScheduler.from_config(style_pipe.scheduler.config)
    # load_ip_adapter() باید قبل از enable_model_cpu_offload() اجرا بشه، وگرنه
    # اجزای جدیدش بیرون از زنجیرهٔ offload می‌مونن (کرش device=meta).
    style_pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
    style_pipe.set_ip_adapter_scale(0.0)
    style_pipe.enable_model_cpu_offload()
    style_pipe.enable_vae_tiling()

    # نکته: localized_inpaint (پایین‌تر) روی بوم ۵۱۲×۵۱۲ کار می‌کنه؛ SDXL برای
    # وضوح ۱۰۲۴+ تنظیم شده، پس کیفیت inpaint در حالت quality محدودتر از generate
    # است — بدون تست زنده روی Colab، پیاده‌سازی یه مسیر جدا با وضوح بالاتر ریسک
    # ناشناخته اضافه می‌کرد؛ به‌عنوان محدودیت شناخته‌شده مستند شده.
    inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
        "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    inpaint_pipe.enable_model_cpu_offload()
    print(f"✅ {SDXL_CHECKPOINT} ready (slower, best photorealism, Img2Img)")


def ensure_model(name):
    """name: "fast" (Realistic Vision / SD1.5) or "quality" (Juggernaut XL / SDXL).
    فقط یه مجموعه مدل در هر لحظه لود می‌مونه — لود همزمان هر دو از RAM سیستم رایگان
    Colab (نه فقط VRAM) رد می‌شه، چون وزن‌های آفلودشده هم توی CPU RAM جا می‌گیرن."""
    global style_pipe, inpaint_pipe
    name = name if name in ("fast", "quality") else "fast"
    if _current_model["name"] == name:
        return
    if style_pipe is not None or inpaint_pipe is not None:
        print(f"Switching model: {_current_model['name']} -> {name} (unloading previous one)...")
        del style_pipe, inpaint_pipe
        style_pipe = None
        inpaint_pipe = None
        gc.collect()
        torch.cuda.empty_cache()
    if name == "quality":
        _load_sdxl()
    else:
        _load_sd15()
    _current_model["name"] = name


print("✅ Model loader ready — ensure_model('fast') or ensure_model('quality')")


In [ ]:
# لود پیش‌فرض SD1.5 موقع استارت — سریع و سبک، سرور بلافاصله آماده می‌شه.
ensure_model("fast")


## ۵) تشخیص اشیا — YOLOv8 + SAM

In [ ]:
from ultralytics import YOLO

print("Loading YOLOv8...")
# روی CPU، مثل SAM پایین — رقابت با SDXL/ControlNet برای VRAM نمی‌کنه؛ تشخیص
# اشیا روی CPU به‌اندازهٔ کافی سریعه که این تعویض ارزششو داره.
yolo_model = YOLO("yolov8x.pt")
yolo_model.to("cpu")
print("✅ YOLOv8 loaded (CPU)!")


In [ ]:
from segment_anything import sam_model_registry, SamPredictor
import requests

sam_path = os.path.join(CACHE_DIR, "sam_vit_h.pth")

if not os.path.exists(sam_path):
    print("Downloading SAM model... (2-3 minutes)")
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    with open(sam_path + ".part", "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    os.replace(sam_path + ".part", sam_path)
    response.close()
    print("✅ SAM downloaded!")
else:
    print("✅ SAM already exists, skipping download!")

sam = sam_model_registry["vit_h"](checkpoint=sam_path)
sam.to("cpu")  # VRAM رو برای مدل تولید تصویر آزاد نگه می‌داره؛ تشخیص کندتر ولی ایمن‌تر
sam_predictor = SamPredictor(sam)
print("✅ SAM loaded (CPU)!")


## ۶) توابع کمکی — تبدیل تصویر، انتخاب ناحیه، inpaint موضعی

این بخش مستقیماً از معماری v2 تیمیه: ذخیرهٔ تصاویر با `image_id`، و `selection_mask` که چهار
روش انتخاب (`region_id`, `mask`, `bbox`, `point`/`points`) رو به یه ماسک یکسان تبدیل می‌کنه.
`localized_inpaint` فقط دور ناحیهٔ انتخابی رو crop و lanczos-resize می‌کنه، بعد نتیجه رو با
لبهٔ feather شده سر جاش می‌چسبونه — همون چیزی که برای رفع مشکل درز/شبح در Furnish لازم بود.

In [ ]:
import base64, io, hashlib, threading, uuid as _uuid
from collections import OrderedDict
import numpy as np, cv2
from PIL import Image, ImageOps

MODEL_LOCK = threading.RLock()
IMAGE_STORE, REGION_STORE = OrderedDict(), OrderedDict()
MAX_IMAGES, MAX_REGION_SETS = 16, 8


def base64_to_pil(value):
    if not isinstance(value, str) or not value:
        raise ValueError("image must be base64")
    try:
        raw = base64.b64decode(value.split(",", 1)[-1], validate=True)
        with Image.open(io.BytesIO(raw)) as im:
            if im.width * im.height > 16_000_000:
                raise ValueError("Image exceeds 16 megapixels")
            return ImageOps.exif_transpose(im).convert("RGB")
    except Exception as exc:
        raise ValueError("Invalid image; use PNG or JPEG, max 16 megapixels") from exc


def pil_to_base64(im):
    b = io.BytesIO()
    im.save(b, format="PNG")
    return base64.b64encode(b.getvalue()).decode()


def image_key(im):
    return hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()


def remember_image(im):
    key = _uuid.uuid4().hex
    IMAGE_STORE[key] = pil_to_base64(im)
    while len(IMAGE_STORE) > MAX_IMAGES:
        IMAGE_STORE.popitem(last=False)
    return {"image_id": key, "image": IMAGE_STORE[key], "mime_type": "image/png",
            "width": im.width, "height": im.height}


def processing_image(im):
    scale = 512 / max(im.size)
    size = tuple(max(1, round(v * scale)) for v in im.size)
    small = im.resize(size, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (512, 512))
    canvas.paste(small, (0, 0))
    a = np.array(canvas)
    if size[0] < 512:
        a[:, size[0]:] = a[:, size[0] - 1:size[0]]
    if size[1] < 512:
        a[size[1]:] = a[size[1] - 1:size[1]]
    return Image.fromarray(a), size


def get_canny_edges(im):
    e = cv2.Canny(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY), 100, 200)
    return Image.fromarray(np.repeat(e[..., None], 3, 2))


def text_prompt(v, name="prompt"):
    if not isinstance(v, str) or not v.strip():
        raise ValueError(name + " is required")
    if len(v) > 600:
        raise ValueError(name + " must be <= 600 characters")
    return v.strip()


def box_pixels(box, im):
    if not isinstance(box, list) or len(box) != 4:
        raise ValueError("bbox must be normalized [x1,y1,x2,y2]")
    x1, y1, x2, y2 = map(float, box)
    if not 0 <= x1 < x2 <= 1 or not 0 <= y1 < y2 <= 1:
        raise ValueError("bbox coordinates must be 0..1")
    return [int(x1 * im.width), int(y1 * im.height),
            min(im.width, int(np.ceil(x2 * im.width))),
            min(im.height, int(np.ceil(y2 * im.height)))]


def _points(values, im):
    if not isinstance(values, list) or not values:
        raise ValueError("positive points required")
    out = []
    for p in values:
        if (not isinstance(p, list) or len(p) != 2
                or not all(isinstance(x, (int, float)) and 0 <= x <= 1 for x in p)):
            raise ValueError("points must be normalized [x,y]")
        out.append([min(im.width - 1, p[0] * im.width), min(im.height - 1, p[1] * im.height)])
    return out


def sam_points_mask(im, positive, negative=None, bbox=None):
    pos = _points(positive, im)
    neg = [] if not negative else _points(negative, im)
    sam_predictor.set_image(np.array(im))
    m, s, _ = sam_predictor.predict(
        point_coords=np.array(pos + neg, np.float32),
        point_labels=np.array([1] * len(pos) + [0] * len(neg)),
        box=np.array(box_pixels(bbox, im), np.float32) if bbox else None,
        multimask_output=True,
    )
    return m[int(np.argmax(s))]


def selection_mask(im, selection):
    if not isinstance(selection, dict):
        raise ValueError("selection is required")
    kinds = [k for k in ("region_id", "mask", "bbox", "point", "points") if selection.get(k) is not None]
    if len(kinds) != 1:
        raise ValueError("Choose one selection type")
    k = kinds[0]
    if k == "region_id":
        item = REGION_STORE.get(image_key(im), {}).get(selection[k])
        if item is None:
            raise ValueError("Region expired; use its mask or detect again")
        mask = item["mask"].copy()
    elif k == "mask":
        v = base64_to_pil(selection[k]).convert("L")
        if v.size != im.size:
            raise ValueError("mask dimensions must match image")
        mask = np.array(v) >= 128
    elif k == "bbox":
        x1, y1, x2, y2 = box_pixels(selection[k], im)
        mask = np.zeros((im.height, im.width), bool)
        mask[y1:y2, x1:x2] = 1
    elif k == "point":
        mask = sam_points_mask(im, [selection[k]])
    else:
        spec = selection[k]
        if not isinstance(spec, dict):
            raise ValueError("points must be an object")
        mask = sam_points_mask(im, spec.get("positive"), spec.get("negative"), spec.get("bbox"))
    if not mask.any():
        raise ValueError("Selection is empty")
    return mask


NEGATIVE = "blurry, distorted, watermark, changed architecture, extra objects"


def localized_inpaint(im, mask, prompt, seed=42, negative=NEGATIVE, expand_px=10, feather_px=5):
    raw = mask.astype("uint8") * 255
    if expand_px:
        k = 2 * expand_px + 1
        raw = cv2.dilate(raw, np.ones((k, k), np.uint8))
    ys, xs = np.where(raw > 0)
    if not len(xs):
        raise ValueError("Selection is empty")
    x1, x2, y1, y2 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    span = max(x2 - x1, y2 - y1)
    pad = max(24, int(span * .38))
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    side = max(128, span + 2 * pad)
    left = max(0, cx - side // 2)
    top = max(0, cy - side // 2)
    right = min(im.width, left + side)
    bottom = min(im.height, top + side)
    left = max(0, right - side)
    top = max(0, bottom - side)

    crop = im.crop((left, top, right, bottom))
    mc = Image.fromarray(raw).crop((left, top, right, bottom))
    canvas, size = processing_image(crop)

    mask512 = Image.new("L", (512, 512))
    mask512.paste(mc.resize(size, Image.Resampling.NEAREST), (0, 0))

    out = inpaint_pipe(
        prompt=prompt, negative_prompt=negative, image=canvas, mask_image=mask512,
        height=512, width=512, strength=.98, num_inference_steps=45, guidance_scale=9,
        generator=torch.Generator(device="cuda").manual_seed(seed),
    ).images[0]

    gen = out.crop((0, 0, *size)).resize(crop.size, Image.Resampling.LANCZOS)
    alpha = np.array(mc)
    if feather_px:
        alpha = cv2.GaussianBlur(alpha, (0, 0), feather_px / 2)

    final = im.copy()
    final.paste(Image.composite(gen, crop, Image.fromarray(alpha)), (left, top))
    return final


print("✅ Helper functions ready (image store, selection_mask, localized_inpaint)")


## ۷) پرامپت‌های ۸ سبک طراحی

In [ ]:
STYLE_PROMPTS = {
    "minimalist": {
        "prompt": "minimalist interior design, pure white and warm beige walls, polished concrete or light wood floor, clean geometric lines, no clutter, abundant natural daylight, recessed ceiling lights, hidden storage, Scandinavian and Japanese wabi-sabi influence, breathing space, monochromatic white and beige palette, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "cluttered, colorful, busy, ornate, dark, multiple patterns, too many objects, cheap, dirty, low quality, blurry, grainy, distorted, watermark, ugly, oversaturated"
    },
    "industrial": {
        "prompt": "industrial interior design, raw exposed red brick walls, polished concrete floor, black steel window frames, exposed metal ceiling beams, Edison bulb pendant lights, vintage leather accents, reclaimed dark wood and iron furniture, urban warehouse aesthetic, raw metal pipes visible, matte black hardware, leather and metal textures, moody warm lighting, Chicago loft style, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "soft, pastel, floral, traditional, plastic, cheap, colorful, bright white, polished, fancy, ornate, blurry, low quality, grainy, watermark"
    },
    "cyberpunk": {
        "prompt": "cyberpunk interior design, dark charcoal walls with hexagonal panels, RGB LED strip lighting glowing blue and purple, neon pink and cyan signs on wall, holographic display panels, carbon fiber textures on furniture, metallic chrome accents, city skyline visible through large window at night with rain, sleek dark furniture with glowing edges, tech gadgets, futuristic floating shelves, neon reflections on floor, Blade Runner inspired, cinematic lighting, ultra sharp, 8k resolution, photorealistic",
        "negative": "foggy, hazy, too dark, obscured furniture, blurry, low quality, traditional, wooden, natural, daytime, bright white, plain walls, no tech elements, watermark, grainy"
    },
    "modern_luxury": {
        "prompt": "ultra luxury modern interior design, Calacatta marble accent wall with gold veining, herringbone light oak hardwood floor, upholstered furniture in cream boucle fabric, sculptural gold brass chandelier, floor to ceiling silk curtains in ivory, marble surfaces with gold legs, designer artwork in gold frames, cashmere throw accents, fresh white orchids in crystal vase, hidden ambient lighting in ceiling coves, five star hotel suite quality, Versace and Fendi inspired, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "cheap, basic, plastic, clutter, industrial, rustic, dark, crowded, low budget, synthetic materials, blurry, grainy, low quality, watermark, distorted"
    },
    "scandinavian": {
        "prompt": "Scandinavian hygge interior design, white washed pine plank floors, pale sage green accent wall, natural oak surfaces, soft wool throws in oatmeal color, rattan pendant light, potted fiddle leaf fig plant, sheepskin rug, floating oak shelves with ceramic vases, linen curtains filtering soft morning light, dried pampas grass in terracotta pot, candles on windowsill, simple geometric cushions in muted tones, Copenhagen apartment style, cozy and warm atmosphere, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "dark, ornate, cluttered, colorful, heavy patterns, gilded, industrial, cold, sterile, cheap, plastic, blurry, low quality, grainy, watermark"
    },
    "midcentury_modern": {
        "prompt": "mid century modern interior design, warm walnut teak wood furniture, geometric patterned wool rug in orange and brown, Eames style lounge chair, tulip side table, sunburst wall clock in gold, abstract 1960s artwork, warm Edison bulb floor lamp with tripod legs, avocado green accent wall, tapered furniture legs, atomic age decorative objects, teak credenza, retro record player on shelf, warm amber lighting, Mad Men inspired, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "contemporary, futuristic, traditional, ornate, dark, gothic, cold colors, cheap, plastic, modern minimalist, blurry, low quality, grainy, watermark"
    },
    "japanese_zen": {
        "prompt": "Japanese zen interior design, natural tatami-textured flooring, shoji screen panels with warm backlight, natural unfinished hinoki wood walls, bamboo ceiling accents, bonsai tree on wooden stand, smooth river stones arrangement, ikebana flower arrangement in ceramic vase, washi paper pendant lamp, moss garden view through window, earthy tones of sand beige and forest green, negative space philosophy, wabi-sabi imperfection, Kyoto ryokan inspired, ultra sharp, 8k resolution, photorealistic, professional interior photography",
        "negative": "cluttered, colorful, western, modern tech, busy patterns, gold, ornate, plastic, synthetic, noisy, loud, western furniture, blurry, low quality, grainy, watermark"
    },
    "bohemian": {
        "prompt": "bohemian interior design, terracotta painted walls, layered Persian and Moroccan rugs on wooden floor, macrame wall hanging, hanging rattan egg chair accent, collection of trailing plants in ceramic and woven pots, warm string fairy lights, gallery wall of eclectic vintage art, colorful embroidered cushions stacked high, suzani throw accents, beaded curtains, incense holder on vintage surface, warm golden hour lighting, Marrakech riad inspired, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "minimal, plain, cold, sterile, corporate, modern sleek, industrial, white walls, no plants, sparse, empty, blurry, low quality, grainy, watermark"
    }
}

# روی همهٔ سبک‌ها اعمال می‌شه تا کارکرد واقعی اتاق عوض نشه (مثلاً آشپزخانه باید
# آشپزخانه بمونه — نسخهٔ اولیهٔ پرامپت‌ها همه‌جا "bedroom" می‌گفت و هر اتاقی رو به
# اتاق‌خواب تبدیل می‌کرد؛ نسخهٔ تیمی هم همین باگ رو داشت، اینجا دوباره برنگشت).
ROOM_PRESERVE_SUFFIX = ", keep the exact same room type and function, same walls, same windows, same doors, same fixed furniture layout — only change wall and floor materials, colors, lighting fixtures and decor"
ROOM_PRESERVE_NEGATIVE = ", different room type, converted room, added bed, added bedroom furniture, structural changes, moved walls, moved windows, moved doors"

# رایج‌ترین خرابی مدل‌ها، ساختار نیست (اون کار ControlNet است) — گرایش به حالت
# نقاشی/رندر به‌جای عکس واقعیه.
QUALITY_SUFFIX = ", RAW photo, shot on DSLR, natural photography, professional real estate photography, realistic materials and textures"
QUALITY_NEGATIVE = ", illustration, painting, drawing, anime, cartoon, 3d render, cgi, video game, plastic looking, artificial, oversaturated"

print("✅ Style prompts ready:", list(STYLE_PROMPTS.keys()))


## ۸) تشخیص اشیا — YOLO + SAM + SegFormer

ترکیب سه‌گانه از معماری v2 تیمی: YOLO جعبه‌ها رو پیدا می‌کنه، SAM هر جعبه رو به ماسک دقیق تبدیل
می‌کنه، و SegFormer (segmentation معنایی صحنه) نواحی‌ای مثل دیوار/کف/سقف که YOLO تشخیص نمی‌ده رو
اضافه می‌کنه. هر شیء یه `region_id` جدا می‌گیره — یعنی اگه دو تا صندلی هم‌شکل تو عکس باشن، می‌شه
فقط یکیشون رو انتخاب و ویرایش کرد.

In [ ]:
_semantic_processor = _semantic_model = None


def semantic_regions(im):
    global _semantic_processor, _semantic_model
    if _semantic_model is None:
        from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
        model_id = "nvidia/segformer-b0-finetuned-ade-512-512"
        _semantic_processor = AutoImageProcessor.from_pretrained(model_id, cache_dir=CACHE_DIR)
        _semantic_model = SegformerForSemanticSegmentation.from_pretrained(model_id, cache_dir=CACHE_DIR).eval()
    inputs = _semantic_processor(images=im, return_tensors="pt")
    with torch.inference_mode():
        output = _semantic_model(**inputs)
    labels = _semantic_processor.post_process_semantic_segmentation(
        output, target_sizes=[(im.height, im.width)])[0].cpu().numpy()
    for cls in np.unique(labels):
        label = _semantic_model.config.id2label[int(cls)].split(";")[0]
        n, parts, stats, _ = cv2.connectedComponentsWithStats((labels == cls).astype("uint8"), 8)
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= max(64, im.width * im.height * 0.003):
                yield label, parts == i


def detect_objects(image_b64):
    im = base64_to_pil(image_b64)
    key, items, public = image_key(im), {}, []

    def add(label, mask, source, confidence=None):
        if len(items) >= 80 or not mask.any():
            return
        ys, xs = np.where(mask)
        rid = _uuid.uuid4().hex
        items[rid] = {"label": label, "mask": mask}
        public.append({
            "id": rid, "label": label, "source": source, "confidence": confidence,
            "bbox": [float(xs.min() / im.width), float(ys.min() / im.height),
                     float((xs.max() + 1) / im.width), float((ys.max() + 1) / im.height)],
            "mask": pil_to_base64(Image.fromarray(mask.astype("uint8") * 255)),
            "mask_mime_type": "image/png",
        })

    with MODEL_LOCK:
        sam_predictor.set_image(np.array(im))
        for result in yolo_model(np.array(im), verbose=False, conf=0.25, max_det=40):
            for box in result.boxes:
                masks, scores, _ = sam_predictor.predict(box=box.xyxy[0].cpu().numpy(), multimask_output=True)
                add(yolo_model.names[int(box.cls.item())], masks[int(np.argmax(scores))], "yolo_sam", float(box.conf.item()))
        # لیبل‌ها و نواحی دیگهٔ ADE20K مکمل تشخیص‌های YOLO/COCO هستن.
        for label, mask in semantic_regions(im):
            if any(label == item["label"] and (mask & item["mask"]).sum() / max(1, (mask | item["mask"]).sum()) > 0.65
                   for item in items.values()):
                continue
            add(label, mask, "segformer")
        REGION_STORE[key] = items
        while len(REGION_STORE) > MAX_REGION_SETS:
            REGION_STORE.popitem(last=False)

    return {"objects": list(dict.fromkeys(x["label"] for x in public)), "regions": public,
            "width": im.width, "height": im.height, "coordinates": "normalized", "image_hash": key}


print("✅ detect_objects ready (YOLO + SAM + SegFormer)")


## ۹) تولید سبک — Img2Img از روی عکس واقعی

In [ ]:
_IP_ADAPTER_NEUTRAL_IMAGE = Image.new("RGB", (224, 224), (128, 128, 128))


def generate_style(image_b64, style_name=None, palette=None, custom_prompt=None, model="fast", seed=42):
    ensure_model(model)
    params = GEN_PARAMS.get(model, GEN_PARAMS["fast"])

    if custom_prompt:
        p = text_prompt(custom_prompt, "customPrompt") + ROOM_PRESERVE_SUFFIX + \
            ", interior design photography, 8k, photorealistic, sharp focus, professional photography, highly detailed"
        neg = "blurry, low quality, distorted, ugly, watermark" + ROOM_PRESERVE_NEGATIVE
    elif style_name in STYLE_PROMPTS:
        config = STYLE_PROMPTS[style_name]
        p = config["prompt"] + ROOM_PRESERVE_SUFFIX
        neg = config["negative"] + ROOM_PRESERVE_NEGATIVE
    else:
        raise ValueError("Choose a valid style or customPrompt")

    if palette:
        if not isinstance(palette, dict):
            raise ValueError("palette must be an object")
        palette_prompt = palette.get("prompt", "")
        colors = palette.get("colors", [])
        if palette_prompt and not isinstance(palette_prompt, str):
            raise ValueError("palette.prompt must be text")
        if colors and (not isinstance(colors, list) or not all(isinstance(c, str) for c in colors)):
            raise ValueError("palette.colors must be a list")
        color_text = ", ".join(colors)
        palette_text = ", ".join(x for x in [palette_prompt.strip(), color_text] if x)
        if palette_text:
            p = palette_text + ", " + p + ", dominant color scheme must follow the selected palette"
            neg += ", wrong colors, different color palette, ignore selected palette"

    original = base64_to_pil(image_b64)
    size = original.size
    room = original.resize((768, 768), Image.Resampling.LANCZOS)
    edge_image = get_canny_edges(room)

    with MODEL_LOCK:
        style_pipe.set_ip_adapter_scale(0.0)
        r = style_pipe(
            prompt=p + QUALITY_SUFFIX + ", highly detailed, 8k, photorealistic, sharp focus, professional photography",
            negative_prompt=neg + QUALITY_NEGATIVE + ", blurry, low quality, distorted, ugly, watermark",
            image=room,               # عکس واقعی اتاق — Img2Img از پیکسل واقعی شروع می‌کنه
            control_image=edge_image,  # نقشهٔ لبه — ساختار رو هم‌راستا نگه می‌داره
            strength=0.7,               # چقدر اجازه داره عکس واقعی تغییر کنه
            ip_adapter_image=_IP_ADAPTER_NEUTRAL_IMAGE,
            num_inference_steps=40,
            guidance_scale=params["guidance_scale"],
            controlnet_conditioning_scale=params["controlnet_conditioning_scale"],
            height=768, width=768,
            generator=torch.Generator(device="cuda").manual_seed(seed),
        ).images[0]

    return {"image": pil_to_base64(r.resize(size, Image.Resampling.LANCZOS)), "mime_type": "image/png"}


def generate_all_previews(image_b64, palette=None, styles=None, model="fast", seed=42):
    styles = list(STYLE_PROMPTS) if styles is None else styles
    if not isinstance(styles, list) or not 1 <= len(styles) <= 8 or any(s not in STYLE_PROMPTS for s in styles):
        raise ValueError("styles must contain 1..8 valid names")
    return {
        "previews": {s: generate_style(image_b64, s, palette=palette, model=model, seed=seed)["image"] for s in styles},
        "mime_type": "image/png",
    }


print("✅ generate_style / generate_all_previews ready (model=\"fast\"/\"quality\")")


## ۱۰) ویرایش، حذف و مبله‌کردن — انتخاب دقیق ناحیه

از معماری v2: `selection` می‌تونه `region_id` (از یه تشخیص قبلی)، `mask`، `bbox` یا `point` باشه.
برای سازگاری با فرانت فعلی (که هنوز UI انتخاب ناحیه نداره)، اگه بجاش فقط اسم شیء (`object`) بفرستی،
خودش یه تشخیص می‌زنه و اولین ناحیهٔ هم‌نام رو انتخاب می‌کنه.

In [ ]:
def edit_object(image_b64, object_label=None, edit_prompt=None, selection=None, delete=False, model="fast", seed=42):
    ensure_model(model)
    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        mask = selection_mask(im, selection)
        if delete:
            p = ("clean continuous background replacing the removed object, "
                 "extend surrounding wall, floor and textures, "
                 "same perspective and lighting, empty selected area")
            if edit_prompt:
                p += ", " + text_prompt(edit_prompt)
            neg = "object, furniture, ghost object, duplicate, blurry, seams, watermark"
            r = localized_inpaint(im, mask, p, seed, neg, 14, 7)
        else:
            p = text_prompt(edit_prompt) + ", replace selected object only, complete object, realistic contact shadow, match room perspective, scale, lighting and surrounding style" + QUALITY_SUFFIX
            neg = "duplicate object, old object, partial object, floating, deformed, blurry, seams, watermark" + QUALITY_NEGATIVE
            r = localized_inpaint(im, mask, p, seed, neg, 10, 5)
    return {"image": pil_to_base64(r), "mime_type": "image/png"}


def resolve_selection(image_b64, data):
    """selection صریح رو برمی‌گردونه، یا اگه فقط اسم شیء (object) داده شده بود،
    یه تشخیص می‌زنه و اولین ناحیهٔ هم‌نام رو به یه region_id تبدیل می‌کنه —
    برای سازگاری با فرانتی که هنوز UI انتخاب ناحیه نداره."""
    selection = data.get("selection")
    if selection:
        return selection
    label = (data.get("object") or "").strip().lower()
    if not label:
        raise ValueError("selection or object is required")
    det = detect_objects(image_b64)
    match = next((r for r in det["regions"] if label in r["label"].lower()), None)
    if match is None:
        raise ValueError(f'No detected object matches "{label}"; try detect-objects first and pass a selection')
    return {"region_id": match["id"]}


def default_furnish_mask(im):
    m = np.zeros((im.height, im.width), bool)
    m[int(im.height * .43):, :] = 1
    return m


def furnish_room(image_b64, furnish_prompt, selection=None, model="fast", seed=42):
    ensure_model(model)
    p = text_prompt(furnish_prompt)
    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        m = default_furnish_mask(im) if selection is None else selection_mask(im, selection)
        r = localized_inpaint(
            im, m,
            p + ", add only requested furniture, preserve existing room style, architecture and unrequested objects, realistic placement, perspective, scale and contact shadows" + QUALITY_SUFFIX,
            seed, "changed room style, changed walls, changed windows, duplicate furniture, floating object, blurry, watermark" + QUALITY_NEGATIVE, 4, 5,
        )
    return {"image": pil_to_base64(r), "mime_type": "image/png"}


# اسم قدیمی که روت‌های /colab-furnish بهش رفرنس می‌دن.
def furnish_room_inpaint(image_b64, furnish_prompt, selection=None, model="fast", seed=42):
    return furnish_room(image_b64, furnish_prompt, selection, model, seed)


print("✅ edit_object / furnish_room / delete (via edit_object(delete=True)) ready")


## ۱۱) افزودن شیء از روی عکس مرجع — IP-Adapter

این تابع تو نسخهٔ تیمی اصلاً وجود نداشت؛ همون قابلیتیه که قبلاً ساختیم: به‌جای توصیف متنیِ شیء،
عکس واقعی‌ش رو می‌گیره (IP-Adapter) و با شدت پایین Img2Img (`strength=0.4`) طوری تو اتاق جا می‌کنه
که بقیهٔ اتاق تقریباً دست‌نخورده بمونه.

In [ ]:
# GEN_PARAMS (controlnet_conditioning_scale 1.05/0.5) is tuned for style
# transfer, where the whole point is staying close to the room's edges. Object
# insertion needs the opposite: ControlNet locked that tight, combined with
# strength=0.4 and the room-preserve prompt, was fighting the IP-Adapter so
# hard that nothing actually got painted in — just faint drift artifacts
# (that ceiling swirl). Loosened specifically for this function.
ADD_OBJECT_PARAMS = {
    "fast":    {"guidance_scale": 8.0, "controlnet_conditioning_scale": 0.4},
    "quality": {"guidance_scale": 6.5, "controlnet_conditioning_scale": 0.3},
}


def add_object_from_reference(room_image_b64, object_image_b64, placement_prompt="", model="fast", seed=42):
    ensure_model(model)
    params = ADD_OBJECT_PARAMS.get(model, ADD_OBJECT_PARAMS["fast"])

    room_image = base64_to_pil(room_image_b64).resize((768, 768), Image.Resampling.LANCZOS)
    object_image = base64_to_pil(object_image_b64)
    edge_image = get_canny_edges(room_image)

    base_prompt = (
        (placement_prompt.strip() or "place this exact item naturally in the room") +
        ", the item from the reference image is clearly visible and present in the scene" +
        ", seamlessly blended, matching perspective and lighting, photorealistic, 8k, interior design photography, highly detailed"
    )
    negative_prompt = (
        "missing object, no new object, unchanged room, floating object, wrong perspective, "
        "mismatched lighting, blurry, low quality, distorted, watermark"
    )

    with MODEL_LOCK:
        style_pipe.set_ip_adapter_scale(0.85)  # کشش قوی‌تر به سمت شیء مرجع
        r = style_pipe(
            prompt=base_prompt + QUALITY_SUFFIX,
            negative_prompt=negative_prompt + QUALITY_NEGATIVE,
            image=room_image,           # عکس واقعی اتاق
            control_image=edge_image,    # حفظ چیدمان/ساختار — ولی شل‌تر از حالت style transfer
            strength=0.55,                 # قبلاً 0.4 بود — خیلی کم بود که شیء واقعاً اضافه بشه
            ip_adapter_image=object_image,
            num_inference_steps=40,
            guidance_scale=params["guidance_scale"],
            controlnet_conditioning_scale=params["controlnet_conditioning_scale"],
            height=768, width=768,
            generator=torch.Generator(device="cuda").manual_seed(seed),
        ).images[0]
        style_pipe.set_ip_adapter_scale(0.0)

    return {"image": pil_to_base64(r), "mime_type": "image/png"}


print("✅ add_object_from_reference ready (model=\"fast\"/\"quality\") — looser ControlNet grip so the object actually appears")


## ۱۲) سرور Flask

روت‌های جدید (`/generate`, `/detect-objects`, `/edit-object`, `/delete-object`, `/furnish-room`,
`/preview-styles`, `/add-object`, `/capabilities`) به‌علاوهٔ نام‌های قدیمی (`/colab-*`) برای سازگاری
کامل با فرانت فعلی. تمام درخواست‌ها (به‌جز OPTIONS) باید هدر `Authorization: Bearer CONNECTION_KEY`
داشته باشن.

In [ ]:
import threading as _threading
import secrets, hmac
from flask import Flask, request, jsonify
from werkzeug.exceptions import HTTPException

colab_app = Flask(__name__)
colab_app.config["MAX_CONTENT_LENGTH"] = 32 * 1024 * 1024
CONNECTION_KEY = secrets.token_urlsafe(32)


@colab_app.after_request
def add_cors(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning, Authorization"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
    return response


@colab_app.before_request
def check_auth():
    if request.method == "OPTIONS":
        resp = colab_app.make_response("")
        resp.headers["Access-Control-Allow-Origin"] = "*"
        resp.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning, Authorization"
        resp.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
        resp.status_code = 200
        return resp
    if not hmac.compare_digest(request.headers.get("Authorization", ""), "Bearer " + CONNECTION_KEY):
        return jsonify({"error": "Connection key required"}), 401


@colab_app.errorhandler(ValueError)
def invalid_input(exc):
    return jsonify({"error": str(exc)}), 400


@colab_app.errorhandler(Exception)
def request_error(exc):
    if isinstance(exc, HTTPException):
        return jsonify({"error": exc.description}), exc.code
    colab_app.logger.exception("Backend request failed")
    return jsonify({"error": f"{type(exc).__name__}: {exc}"}), 500


def payload():
    data = request.get_json(silent=True)
    if not isinstance(data, dict):
        raise ValueError("JSON object required")
    return data


def request_image(data):
    if data.get("image"):
        return pil_to_base64(base64_to_pil(data["image"]))
    key = data.get("image_id")
    if not isinstance(key, str) or key not in IMAGE_STORE:
        raise ValueError("Send image or a valid image_id from /upload; expired images must be uploaded again")
    return IMAGE_STORE[key]


def request_seed(data):
    value = data.get("seed", 42)
    if type(value) is not int or not 0 <= value < 2 ** 32:
        raise ValueError("seed must be an integer 0..4294967295")
    return value


def request_model(data):
    return data.get("model") if data.get("model") in ("fast", "quality") else "fast"


def finish(result):
    if "image" in result:
        result.update(remember_image(base64_to_pil(result["image"])))
    return jsonify(result)


@colab_app.get("/health")
def health():
    return jsonify({"status": "ok", "colab_connected": True, "mode": "colab",
                     "api_version": 2, "current_model": _current_model["name"]})


@colab_app.get("/capabilities")
def capabilities():
    return jsonify({
        "api_version": 2, "styles": list(STYLE_PROMPTS), "models": ["fast", "quality"],
        "current_model": _current_model["name"],
        "operations": ["style", "furnish", "detect", "edit", "delete", "add-object"],
        "selection": ["region_id", "mask", "bbox", "point", "points"],
        "coordinates": "normalized", "furnish_requires_selection": False,
        "output_mime_type": "image/png",
    })


@colab_app.post("/upload")
def upload():
    if "image" not in request.files:
        raise ValueError("image file required")
    im = base64_to_pil(base64.b64encode(request.files["image"].read()).decode())
    with MODEL_LOCK:
        return jsonify(remember_image(im))


@colab_app.post("/generate")
@colab_app.post("/colab-generate")
def generate():
    data = payload()
    with MODEL_LOCK:
        return finish(generate_style(request_image(data), data.get("style"), data.get("palette"),
                                      data.get("customPrompt"), request_model(data), request_seed(data)))


@colab_app.post("/detect-objects")
@colab_app.post("/colab-detect")
def detect_objects_route():
    data = payload()
    with MODEL_LOCK:
        return jsonify(detect_objects(request_image(data)))


@colab_app.post("/segment-point")
def segment_point_route():
    data = payload()
    with MODEL_LOCK:
        im = base64_to_pil(request_image(data))
        mask = selection_mask(im, {"point": data.get("point")})
        key, rid = image_key(im), _uuid.uuid4().hex
        regions = REGION_STORE.setdefault(key, {})
        if len(regions) >= 80:
            raise ValueError("Too many regions; run detection again")
        regions[rid] = {"label": "selected region", "mask": mask}
        while len(REGION_STORE) > MAX_REGION_SETS:
            REGION_STORE.popitem(last=False)
        return jsonify({"region_id": rid, "image_hash": key,
                         "mask": pil_to_base64(Image.fromarray(mask.astype("uint8") * 255)),
                         "width": im.width, "height": im.height, "mask_mime_type": "image/png"})


@colab_app.post("/segment-points")
def segment_points_route():
    data = payload()
    with MODEL_LOCK:
        im = base64_to_pil(request_image(data))
        mask = selection_mask(im, {"points": {"positive": data.get("positive"),
                                               "negative": data.get("negative", []),
                                               "bbox": data.get("bbox")}})
        return jsonify({"mask": pil_to_base64(Image.fromarray(mask.astype("uint8") * 255)),
                         "width": im.width, "height": im.height, "mask_mime_type": "image/png"})


@colab_app.post("/edit-object")
@colab_app.post("/colab-edit")
def edit_object_route():
    data = payload()
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(edit_object(image_b64, data.get("object"), data.get("prompt"), selection,
                                   delete=False, model=request_model(data), seed=request_seed(data)))


@colab_app.post("/delete-object")
@colab_app.post("/colab-delete")
def delete_object_route():
    data = payload()
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(edit_object(image_b64, data.get("object"), data.get("prompt"), selection,
                                   delete=True, model=request_model(data), seed=request_seed(data)))


@colab_app.post("/furnish-room")
@colab_app.post("/colab-furnish")
def colab_furnish():
    data = payload()
    with MODEL_LOCK:
        return finish(furnish_room(request_image(data), data.get("prompt"), data.get("selection"),
                                    request_model(data), request_seed(data)))


@colab_app.post("/preview-styles")
@colab_app.post("/colab-preview")
def preview_styles_route():
    data = payload()
    with MODEL_LOCK:
        return jsonify(generate_all_previews(request_image(data), data.get("palette"), data.get("styles"),
                                              request_model(data), request_seed(data)))


@colab_app.post("/add-object")
@colab_app.post("/colab-add-object")
def add_object_route():
    data = payload()
    room_image = data.get("room_image")
    object_image = data.get("object_image")
    if not room_image or not object_image:
        raise ValueError("room_image and object_image required")
    with MODEL_LOCK:
        return finish(add_object_from_reference(pil_to_base64(base64_to_pil(room_image)),
                                                 pil_to_base64(base64_to_pil(object_image)),
                                                 data.get("prompt", ""), request_model(data), request_seed(data)))


print("✅ Flask app defined —", len(list(colab_app.url_map.iter_rules())), "routes registered")


## ۱۳) اجرا و اتصال — این سلول را آخر اجرا کن

In [ ]:
from getpass import getpass
from pyngrok import ngrok
from werkzeug.serving import make_server
import logging

logging.getLogger("pyngrok").setLevel(logging.CRITICAL)
ngrok.set_auth_token(getpass("Your ngrok authtoken (hidden): "))

server = make_server("127.0.0.1", 7860, colab_app, threaded=True)
_threading.Thread(target=server.serve_forever, daemon=True).start()

try:
    tunnel = ngrok.connect(7860)
except Exception:
    raise RuntimeError("Tunnel failed. Check your own ngrok authtoken and account.") from None

print("BACKEND_URL:", tunnel.public_url)
print("CONNECTION_KEY:", CONNECTION_KEY)
print("این دو مقدار رو تو مودال «Connect AI Backend» فرانت وارد کن. رانتایم رو زنده نگه دار.")


## 🎉 تمام شد

معماری این نسخه (انتخاب دقیق ناحیه، ذخیرهٔ تصویر با شناسه، حذف شیء، تشخیص ترکیبی) از نوت‌بوک
بازنویسی‌شدهٔ هم‌گروهی گرفته شده؛ مدل‌ها، Img2Img، افزودن شیء از روی عکس مرجع، و امنیت اتصال از
کار قبلی خودمون. فرانت فعلی بدون تغییر باهاش کار می‌کنه.